In [20]:
# Instalar Whisper, numpy 1.26.4 y FFmpeg (requiere git y ffmpeg ya instalados)
!pip install openai-whisper --quiet
!pip install numpy==1.26.4 --quiet
!pip install ffmpeg-python noisereduce transformers librosa jiwer --quiet

In [21]:
import numpy
print(numpy.__version__)

1.26.4


In [22]:
import noisereduce as nr
import soundfile as sf
import numpy as np
import librosa

AUDIO_INPUT = "audiotest.wav"  # Asegúrate de que exista

# Leer el audio
audio, sr = sf.read(AUDIO_INPUT)

# Convertir a mono si es estéreo
if len(audio.shape) == 2:
    print("🎧 Audio estéreo detectado. Convirtiendo a mono...")
    audio = np.mean(audio, axis=1)

# Normalizar volumen
audio = audio / np.max(np.abs(audio))

# Aplicar reducción de ruido
print(f"🔇 Aplicando reducción de ruido, duración: {len(audio)/sr:.2f} segundos...")
cleaned_audio = nr.reduce_noise(y=audio, sr=sr)

# Guardar el audio limpio
AUDIO_CLEAN = "audio_clean.wav"
sf.write(AUDIO_CLEAN, cleaned_audio, sr)
print(f"✅ Audio limpio guardado en {AUDIO_CLEAN}")


🎧 Audio estéreo detectado. Convirtiendo a mono...
🔇 Aplicando reducción de ruido, duración: 120.96 segundos...
✅ Audio limpio guardado en audio_clean.wav


In [23]:
import os
import subprocess

AUDIO_PATH = "audio_clean.wav"

assert os.path.exists(AUDIO_PATH), f"❌ El archivo {AUDIO_PATH} no se encuentra."

# Ver propiedades del audio
subprocess.run(["ffmpeg", "-i", AUDIO_PATH])


CompletedProcess(args=['ffmpeg', '-i', 'audio_clean.wav'], returncode=1)

In [24]:
import whisper

model = whisper.load_model("large-v2")

result = model.transcribe(
    AUDIO_PATH,
    language="es",
    fp16=False,
    verbose=True,
    condition_on_previous_text=True,
    temperature=0.0,
    best_of=5
)

print("📝 Transcripción completa:")
print(result['text'])


[00:00.000 --> 00:02.000]  Qué bien está de nido, ¿eh?
[00:02.000 --> 00:04.000]  A Berlín eran 36 horas en autobús
[00:04.000 --> 00:07.000]  y un currito del hotel me habló de un negocio
[00:07.000 --> 00:09.000]  turco, feo como un demonio.
[00:09.000 --> 00:11.000]  Partía, me echó un polvo.
[00:12.000 --> 00:14.000]  La gente dice, por ahí me hinché a vender citones.
[00:14.000 --> 00:16.000]  Me levanto una mañana y digo
[00:16.000 --> 00:19.000]  coge en la puerta y te vas tú y tu puta guitarra.
[00:20.000 --> 00:23.000]  Yo no creo en Dios, pero sí que hay algo.
[00:23.000 --> 00:25.000]  ¿No has tenido almohadana?
[00:25.000 --> 00:27.000]  Yo soy única de él, ¿no veas qué dolor?
[00:27.000 --> 00:29.000]  Bueno, entonces lo dejamos.
[00:29.000 --> 00:31.000]  Tengo un acuerdo.
[00:31.000 --> 00:33.000]  Y él se intentó suicidar.
[00:33.000 --> 00:35.000]  Pues me volví a España, tía.
[00:35.000 --> 00:37.000]  Y llego y digo, ¿yo qué hago aquí?
[00:37.000 --> 00:39.000]  Yo a

In [25]:
from datetime import timedelta

def format_timestamp(seconds):
    return str(timedelta(seconds=int(seconds))) + "," + str(int((seconds % 1) * 1000)).zfill(3)

def export_srt(segments, filename="audiotest.srt"):
    with open(filename, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments, 1):
            start = format_timestamp(seg['start'])
            end = format_timestamp(seg['end'])
            text = seg['text'].strip()
            f.write(f"{i}\n{start} --> {end}\n{text}\n\n")

export_srt(result["segments"])
print("✅ Subtítulos exportados")


✅ Subtítulos exportados


In [29]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_id = "dreuxx26/Multilingual-grammar-Corrector-using-mT5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def corregir_texto(texto):
    input_text = texto
    #input_text = texto
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = lm_model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Abrimos archivo SRT en modo escritura
with open("audiotest_corregido.srt", "w", encoding="utf-8") as f:
    for i, segment in enumerate(result['segments'], start=1):
        start_time = format_timestamp(segment['start'])
        end_time = format_timestamp(segment['end'])

        print(f"\n⏱️ [{segment['start']:.2f}s - {segment['end']:.2f}s]")
        original = segment['text']
        corregido = corregir_texto(original)
        print(f"Original: {original}")
        print(f"Corregido: {corregido}")

        # Escribir en archivo SRT
        f.write(f"{i}\n{start_time} --> {end_time}\n{corregido.strip()}\n\n")

print("✅ Proceso completado. Subtítulos corregidos guardados como 'audiotest_corregido.srt'")
 

c:\ProgramData\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5TokenizerFast'.
c:\ProgramData\anaconda3\Lib\site-packages\transformers\convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are c


⏱️ [0.00s - 2.00s]
Original:  Qué bien está de nido, ¿eh?
Corregido: Qué bien está de nido, ¿eh?

⏱️ [2.00s - 4.00s]
Original:  A Berlín eran 36 horas en autobús
Corregido: A Berlín eran 36 horas en autobús.

⏱️ [4.00s - 7.00s]
Original:  y un currito del hotel me habló de un negocio
Corregido: y un currito del hotel me habló de un negocio

⏱️ [7.00s - 9.00s]
Original:  turco, feo como un demonio.
Corregido: También feo como un demonio.

⏱️ [9.00s - 11.00s]
Original:  Partía, me echó un polvo.
Corregido: También me echó un polvo.

⏱️ [12.00s - 14.00s]
Original:  La gente dice, por ahí me hinché a vender citones.
Corregido: La gente dice, por ahí me hinché a vender citones.

⏱️ [14.00s - 16.00s]
Original:  Me levanto una mañana y digo
Corregido: Me levanto una mañana y digo.

⏱️ [16.00s - 19.00s]
Original:  coge en la puerta y te vas tú y tu puta guitarra.
Corregido: Tiene tú y tu puta guitarra.

⏱️ [20.00s - 23.00s]
Original:  Yo no creo en Dios, pero sí que hay algo.
Corregido: Yo no